# 79 — Analyze Q-planning fork acquisition pilot

Run after all four notebook-78 workers. The primary question is whether U20-prioritized roots
produce more mixed-outcome trees or more oracle improvement than equal-budget random and
failure-prioritized roots.


## 1. Environment and completeness


In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen(
    'https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/'
    'pnp-vla/scripts/colab_bootstrap.py').read().decode())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pnp.qplanning_fork_pilot import (
    FORK_PILOT_CANDIDATES,
    FORK_PILOT_STRATEGIES,
    FORK_PILOT_TREES_PER_STRATEGY,
    load_fork_pilot_results,
)

trees, summary = load_fork_pilot_results()
display(summary)

completion = (
    trees.groupby('strategy', observed=True)
    .agg(expected_trees=('candidate_group_id', 'size'),
         complete_trees=('complete', 'sum'),
         logged_branches=('branches', 'sum'))
    .reset_index()
)
completion['expected_branches'] = (
    FORK_PILOT_TREES_PER_STRATEGY * FORK_PILOT_CANDIDATES)
display(completion)

assert set(completion.expected_trees) == {FORK_PILOT_TREES_PER_STRATEGY}
assert set(completion.complete_trees) == {FORK_PILOT_TREES_PER_STRATEGY}, (
    'The full pilot is incomplete; inspect the table before interpreting outcomes.')


## 2. Primary acquisition comparison


In [ ]:
order = list(FORK_PILOT_STRATEGIES)
plot = summary.set_index('strategy').loc[order]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(order, plot.mixed_outcome_trees_pct, color=['#4C78A8', '#F58518', '#54A24B'])
axes[0].set(title='Do priorities find decision-sensitive roots?',
            ylabel='Mixed-outcome trees (%)', xlabel='Root-selection strategy')
axes[1].bar(order, plot.oracle_gain_over_stock_pp, color=['#4C78A8', '#F58518', '#54A24B'])
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set(title='Available improvement inside each tree',
            ylabel='Any-branch success minus stock branch (pp)',
            xlabel='Root-selection strategy')
for axis in axes:
    axis.grid(axis='y', alpha=.25)
fig.tight_layout()
plt.show()


## 3. Outcome variation and selected uncertainty


In [ ]:
complete = trees[trees.complete].copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for strategy, group in complete.groupby('strategy', observed=True):
    axes[0].scatter(
        group.source_three_boundary_u20,
        group.branch_success_fraction,
        label=strategy, alpha=.7)
axes[0].set(
    xlabel='Stored three-boundary mean U20 at selected root',
    ylabel='Fraction of nine branches that succeed',
    title='Uncertainty versus branch outcome')
axes[0].legend()
axes[0].grid(alpha=.25)

data = [complete.loc[complete.strategy == strategy,
                     'source_three_boundary_u20'].to_numpy()
        for strategy in order]
axes[1].boxplot(data, tick_labels=order, showfliers=True)
axes[1].set(
    xlabel='Root-selection strategy',
    ylabel='Stored three-boundary mean U20',
    title='Did U20 priority select higher-U roots?')
axes[1].grid(axis='y', alpha=.25)
fig.tight_layout()
plt.show()

suite_table = (
    complete.groupby(['strategy', 'suite'], observed=True)
    .agg(trees=('candidate_group_id', 'size'),
         mixed_pct=('mixed_outcomes', lambda x: 100 * x.mean()),
         stock_sr_pct=('stock_success', lambda x: 100 * x.mean()),
         any_success_pct=('any_success', lambda x: 100 * x.mean()),
         mean_u20=('source_three_boundary_u20', 'mean'))
    .reset_index()
)
display(suite_table)


## 4. Go/no-go interpretation


In [ ]:
random_mixed = float(plot.loc['random', 'mixed_outcome_trees_pct'])
u20_mixed = float(plot.loc['u20', 'mixed_outcome_trees_pct'])
random_gain = float(plot.loc['random', 'oracle_gain_over_stock_pp'])
u20_gain = float(plot.loc['u20', 'oracle_gain_over_stock_pp'])

decision = {
    'u20_vs_random_mixed_ratio': (
        u20_mixed / random_mixed if random_mixed > 0 else np.inf),
    'u20_minus_random_oracle_gain_pp': u20_gain - random_gain,
    'recommended_next_step': (
        'train fork-aware critics with 65% tree replay'
        if (u20_mixed >= 1.5 * random_mixed or u20_gain >= random_gain + 5)
        else 'improve candidate diversity/root selection before critic training'),
}
decision
